# Oocyte Ooplasm Segmentation + Fluorescence Quantification Pipeline

**What this does, per file:**
1. Loads the `.tcf` holotomography (HT) volume for a chosen timepoint (timelapse files
   store multiple timepoints in the same file -- see Part 0 below).
2. Segments the ooplasm (downsampled 2x for memory safety), auto-picking an RI threshold
   via Otsu, floored at `1.3376` -- this floor was chosen by hand-checking several
   thresholds and picking the tightest one that reliably excludes the zona pellucida and
   nearby debris on the original dataset this pipeline was built on. **If you're now
   working with a different dataset/instrument, re-check this floor by eye before
   trusting it** -- see the overlay images.
3. Projects the 3D ooplasm mask to 2D and warps it onto the fluorescence (FL) image grid,
   using the fact that the HT and FL stacks share the same physical stage position/FOV --
   just a resolution-ratio rescale + center alignment, no rotation. **Re-check this
   assumption by eye on a new dataset/instrument setup.**
4. Computes several quantitative measures from the FL signal inside vs. outside the mask
   (see Part 4).

**Before trusting numbers on a new file/batch:** always check the saved overlay image --
the mask should hug the true ooplasm edge, staying inside the zona, without leaking into
debris. Every result includes a saved overlay image so this spot-check stays easy.

---

**New in this version:**
- **Timepoint selection** -- pick which frame of a timelapse to analyze, per file
  (Part 0 and Part 2/3/4).
- **More quantitative measures** -- mean pixel intensity inside the ROI, and a plain
  background-subtracted mean, alongside the existing SNR (Part 4).
- **Results now keyed by (file, timepoint)**, not just file -- so processing the same
  file at two different timepoints doesn't overwrite the first result (Part 5).
- **Graphing helpers** for the results CSV -- bar chart, histogram, scatter of one metric
  vs another, and metric-vs-timepoint trend lines (Part 6).


In [51]:
import numpy as np
import h5py
from scipy import ndimage
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import csv


## Part 1: Loading & metadata

In [52]:
def inspect_tcf_metadata(file_path):
    """Print all attributes found anywhere in the file -- use this on any
    NEW file to confirm resolution / RI scale / registration before trusting
    the defaults baked into the functions below."""
    with h5py.File(file_path, "r") as f:
        def visitor(name, obj):
            if obj.attrs:
                for key, val in obj.attrs.items():
                    print(f"{name} :: {key} = {val}")
        f.visititems(visitor)
        for key, val in f.attrs.items():
            print(f"/ :: {key} = {val}")


def get_voxel_size_and_n_medium(file_path, timepoint=0):
    """Voxel size (dz, dy, dx) in um and a data-derived n_medium (RIMin) from
    the file's own 3D metadata, for the given timepoint."""
    tp_key = f"{timepoint:06d}"
    with h5py.File(file_path, "r") as f:
        data3d_attrs = dict(f["Data"]["3D"].attrs)
        pos_attrs = dict(f["Data"]["3D"][tp_key].attrs)
    voxel_size_um = (
        float(data3d_attrs["ResolutionZ"][0]),
        float(data3d_attrs["ResolutionY"][0]),
        float(data3d_attrs["ResolutionX"][0]),
    )
    n_medium = float(pos_attrs["RIMin"][0])
    return voxel_size_um, n_medium


## Part 0: Checking available timepoints (timelapse files)

Timelapse `.tcf` files store each timepoint as its own zero-padded group
(`000000`, `000001`, `000002`, ...) under `Data/3D` and `Data/3DFL/<channel>`. Every
function below now takes a `timepoint` argument (an integer, 0-indexed) so you can pick
which frame to analyze -- e.g. if you want "image 4" of a 5-image timelapse, that's
`timepoint=3` (0-indexed: image 1 = timepoint 0).

**Not every file will have the same number of timepoints or the same interesting frame**
-- use `list_available_timepoints()` on a new file before picking one, and double check
against the overlay once you've picked it.

In [53]:
def list_available_timepoints(file_path):
    """List the timepoint indices available under Data/3D and Data/3DFL for a file.
    Run this before picking a timepoint on a new/unfamiliar file -- timepoint counts
    and indexing can differ between timelapses.

    Returns (ht_timepoints, fl_timepoints_by_channel).
    """
    with h5py.File(file_path, "r") as f:
        ht_tps = sorted(f["Data"]["3D"].keys())
        fl_tps = {}
        if "3DFL" in f["Data"]:
            for ch in f["Data"]["3DFL"].keys():
                fl_tps[ch] = sorted(f["Data"]["3DFL"][ch].keys())
    print(f"{Path(file_path).name}")
    print(f"  HT timepoints ({len(ht_tps)}): {ht_tps}")
    for ch, tps in fl_tps.items():
        print(f"  FL[{ch}] timepoints ({len(tps)}): {tps}")
    return ht_tps, fl_tps


## Part 2: Ooplasm segmentation (downsampled, memory-safe)

Reads and downsamples the volume in Z-chunks so the full-resolution (~2.9GB) array is
never held in memory at once -- needed because full-resolution segmentation reliably
crashed the kernel on the original workspace this was built on. If you ever move to a
workspace with more memory/resources, `factor=1` would give full-resolution segmentation
instead.

In [54]:
def suggest_ri_threshold_otsu(ri_tomogram, n_bins=256):
    """Otsu threshold, used as an automatic per-file starting point."""
    counts, edges = np.histogram(ri_tomogram, bins=n_bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    total = counts.sum()
    weight_bg = np.cumsum(counts)
    weight_fg = total - weight_bg
    with np.errstate(invalid="ignore", divide="ignore"):
        mean_bg = np.cumsum(counts * centers) / weight_bg
        mean_fg = (np.sum(counts * centers) - np.cumsum(counts * centers)) / weight_fg
        between_class_var = weight_bg * weight_fg * (mean_bg - mean_fg) ** 2
    between_class_var = np.nan_to_num(between_class_var, nan=0.0)
    return float(centers[int(np.argmax(between_class_var))])


def segment_ooplasm_2d_mask(file_path, timepoint=0, outer_threshold=None,
                             threshold_floor=1.3376, factor=2, chunk_z=30,
                             min_size=500, verbose=True):
    """
    Segment the ooplasm in 3D (downsampled, memory-safe) for the given timepoint, then
    project to 2D on the same pixel grid as Data/2DMIP (shares X/Y resolution+size with
    Data/3D).

    outer_threshold : fixed value to use for every call, or None to auto-suggest
        via Otsu on the downsampled volume (floored at threshold_floor -- Otsu
        alone drifted low enough on the original dataset to let the mask leak past the
        true ooplasm edge into zona/debris; the floor was picked by comparing
        several candidate thresholds by eye against the fluorescence overlay).

    Returns (mask_2d_on_ht_grid, threshold_used).
    """
    tp_key = f"{timepoint:06d}"
    with h5py.File(file_path, "r") as f:
        dset = f["Data"]["3D"][tp_key]
        shape = dset.shape
        chunks = []
        for z0 in range(0, shape[0], chunk_z):
            z1 = min(z0 + chunk_z, shape[0])
            chunk = dset[z0:z1].astype(np.float32) / 10000.0
            chunks.append(chunk[::factor, ::factor, ::factor])
            del chunk
        small = np.concatenate(chunks, axis=0)
        ht2d_size = (int(f["Data"]["2DMIP"].attrs["SizeY"][0]),
                     int(f["Data"]["2DMIP"].attrs["SizeX"][0]))

    if outer_threshold is None:
        outer_threshold = suggest_ri_threshold_otsu(small)
        outer_threshold = max(outer_threshold, threshold_floor)
        if verbose:
            print(f"  auto threshold: {outer_threshold:.4f}")

    raw_mask = small > outer_threshold
    labeled, n_components = ndimage.label(raw_mask)
    sizes = ndimage.sum(raw_mask, labeled, range(1, n_components + 1))
    centroids = ndimage.center_of_mass(raw_mask, labeled, range(1, n_components + 1))
    volume_center = np.array(raw_mask.shape) / 2
    candidates = []
    for i, (size, centroid) in enumerate(zip(sizes, centroids)):
        if size > min_size:
            dist = np.linalg.norm(np.array(centroid) - volume_center)
            candidates.append((i + 1, size, dist))
    candidates.sort(key=lambda c: c[1], reverse=True)
    if not candidates:
        raise ValueError(
            f"No candidate component found for '{file_path}' (timepoint={timepoint}) "
            f"at threshold={outer_threshold:.4f}. Try a lower threshold_floor "
            f"or check the file's RI range with inspect_tcf_metadata()."
        )
    if verbose:
        print(f"  {len(candidates)} candidates, chosen size={candidates[0][1]:.0f}")

    chosen = candidates[0][0]
    mask3d = ndimage.binary_closing(labeled == chosen, structure=np.ones((3, 5, 5)))
    for z in range(mask3d.shape[0]):
        mask3d[z] = ndimage.binary_fill_holes(mask3d[z])

    mid_z = mask3d.shape[0] // 2
    mask2d_small = mask3d[mid_z]  # middle slice only
    mask2d_full = ndimage.zoom(mask2d_small.astype(np.uint8), factor, order=0).astype(bool)

    out = np.zeros(ht2d_size, dtype=bool)
    y = min(out.shape[0], mask2d_full.shape[0])
    x = min(out.shape[1], mask2d_full.shape[1])
    out[:y, :x] = mask2d_full[:y, :x]
    return out, outer_threshold


## Part 3: Warp mask onto the fluorescence grid + load FL image

The HT and FL stacks in the original dataset share the same physical stage position and
field of view (confirmed: `PositionX/Y/C` match between `Data/3D` and `Data/3DFL`, and the
computed FOV in microns matches for both) -- so a simple resolution-ratio rescale +
center-crop/pad aligns them correctly. An attempt to instead use the file's own
`Info/MetaData/FL/Registration` transform (Rotation/Scale/Translation) gave a badly
misaligned result on that dataset, so that approach was abandoned in favor of this
simpler, visually-confirmed one. **Always re-check alignment by eye on a new dataset /
instrument setup** before trusting this.

(The resolution/size attributes used here come from `Data/2DMIP` and `Data/2DFLMIP`,
which aren't timepoint-indexed, so this step doesn't need a `timepoint` argument.)

In [55]:
def warp_mask_ht_to_fl(mask_ht_2d, file_path):
    """Center-align and rescale the HT mask onto the FL grid."""
    with h5py.File(file_path, "r") as f:
        res_ht = float(f["Data"]["2DMIP"].attrs["ResolutionX"][0])
        res_fl = float(f["Data"]["2DFLMIP"].attrs["ResolutionX"][0])
        fl_shape = (int(f["Data"]["2DFLMIP"].attrs["SizeY"][0]),
                    int(f["Data"]["2DFLMIP"].attrs["SizeX"][0]))

    zoom_factor = res_ht / res_fl
    scaled = ndimage.zoom(mask_ht_2d.astype(np.uint8), zoom_factor, order=0)

    out = np.zeros(fl_shape, dtype=np.uint8)
    sy, sx = scaled.shape
    oy, ox = fl_shape
    y0 = (sy - oy) // 2
    x0 = (sx - ox) // 2
    if y0 >= 0 and x0 >= 0:
        out[:, :] = scaled[y0:y0 + oy, x0:x0 + ox]
    else:
        y0p, x0p = max(0, -y0), max(0, -x0)
        out[y0p:y0p + sy, x0p:x0p + sx] = scaled

    return out.astype(bool)


def load_fl_2d_mip(file_path, timepoint=0, channel="CH0"):
    """Loads the MIDDLE slice of the 3D fluorescence stack (not a max projection),
    for the given timepoint."""
    tp_key = f"{timepoint:06d}"
    with h5py.File(file_path, "r") as f:
        dset = f["Data"]["3DFL"][channel][tp_key]
        mid_z = dset.shape[0] // 2
        return dset[mid_z].astype(np.float32)


def load_ht_2d_slice(file_path, timepoint=0, factor=2, chunk_z=30):
    """Loads the middle Z-slice of the HT (tomogram) volume for the given timepoint,
    downsampled to match the mask's resolution, for a same-scale overlay."""
    tp_key = f"{timepoint:06d}"
    with h5py.File(file_path, "r") as f:
        dset = f["Data"]["3D"][tp_key]
        mid_z = dset.shape[0] // 2
        img = dset[mid_z].astype(np.float32) / 10000.0
    return img


## Part 4: Quantitative measures + visual check

For the chosen timepoint, three measures are computed from the FL image, all using the
same inside-mask (ROI) vs. outside-mask (background) split:

- **`mean_intensity_roi`** -- mean FL pixel intensity inside the ooplasm mask. The
  simplest measure, with no background correction.
- **`background_subtracted`** -- `mean(inside mask) - mean(outside mask)`. Corrects for
  a uniform background offset, still in raw intensity units.
- **`snr`** -- `(mean(inside) - mean(outside)) / std(outside)`. Same numerator as
  background-subtracted, but scaled by background noise (its std dev), so it's
  comparable across images with different background noise levels.

All three are reported together so you can see how much the picture changes once
background/noise is accounted for.

In [56]:
_MASK_COLOR = mcolors.ListedColormap(["none", "crimson"])
_GREEN_FL_CMAP = mcolors.LinearSegmentedColormap.from_list("black_green", ["black", "lime"])


def compute_snr_one_file(file_path, timepoint=0, condition="", oocyte_id="", time_min=None,
                          outer_threshold=None, fl_channel="CH0", show_overlay=True,
                          save_path=None, verbose=True):
    """
    Full pipeline for ONE file at ONE (internal) timepoint/frame: segment ooplasm ->
    warp onto FL grid -> compute quantitative measures.

    condition : a free-text label for this run, e.g. "MTG_100", "TMRM_60" -- not used
        in any calculation, just carried through into the result/CSV/overlay so you can
        group and compare across conditions (see Part 7).
    oocyte_id : a free-text label identifying which biological oocyte this run belongs
        to, e.g. "MTG100_oocyte1" -- lets you track the SAME oocyte across several runs,
        whether those runs come from separate files or separate frames of one timelapse
        file (see Part 8). Leave blank if you're not tracking oocyte identity.
    time_min : the REAL elapsed imaging time in minutes for this run (e.g. 30, 60, 120),
        independent of the internal `timepoint` frame index -- some of your data is
        separate files per timepoint, some is multiple frames in one timelapse file, so
        the internal frame index alone doesn't tell you the real time. Leave as None if
        not tracking a time series.

    save_path : if given, saves the overlay image there (PNG) instead of/as well as
        displaying it -- use this for batch runs so you can review every file's
        alignment afterward without dozens of inline plots.

    Returns a dict with file, timepoint, condition, oocyte_id, time_min, threshold_used,
    mean_intensity_roi, background_mean, background_subtracted, noise_std, snr.
    """
    mask_ht_2d, threshold_used = segment_ooplasm_2d_mask(
        file_path, timepoint=timepoint, outer_threshold=outer_threshold, verbose=verbose
    )
    mask_fl_2d = warp_mask_ht_to_fl(mask_ht_2d, file_path)
    fl_image = load_fl_2d_mip(file_path, timepoint=timepoint, channel=fl_channel)

    mean_intensity_roi = float(fl_image[mask_fl_2d].mean())
    background_mean = float(fl_image[~mask_fl_2d].mean())
    noise_std = float(fl_image[~mask_fl_2d].std())
    background_subtracted = mean_intensity_roi - background_mean
    snr = background_subtracted / noise_std if noise_std > 0 else float("nan")

    if show_overlay or save_path:
        ht_image = load_ht_2d_slice(file_path, timepoint=timepoint)

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))

        axes[0].imshow(ht_image, cmap="gray")
        axes[0].imshow(mask_ht_2d, cmap=_MASK_COLOR, alpha=0.4)
        axes[0].set_title("HT (tomogram) + mask")
        axes[0].axis("off")

        axes[1].imshow(fl_image, cmap=_GREEN_FL_CMAP)
        axes[1].imshow(mask_fl_2d, cmap=_MASK_COLOR, alpha=0.4)
        axes[1].set_title(
            f"Fluorescence + mask\nSNR={snr:.2f}, bg-sub={background_subtracted:.1f}, "
            f"threshold={threshold_used:.4f}"
        )
        axes[1].axis("off")

        tags = [f"timepoint {timepoint}"]
        if condition:
            tags.append(condition)
        if oocyte_id:
            tags.append(oocyte_id)
        if time_min is not None:
            tags.append(f"{time_min} min")
        fig.suptitle(f"{Path(file_path).name}  ({', '.join(tags)})")
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
        if show_overlay:
            plt.show()
        else:
            plt.close()

    return {
        "file": Path(file_path).name,
        "timepoint": timepoint,
        "condition": condition,
        "oocyte_id": oocyte_id,
        "time_min": time_min,
        "threshold_used": threshold_used,
        "mean_intensity_roi": mean_intensity_roi,
        "background_mean": background_mean,
        "background_subtracted": background_subtracted,
        "noise_std": noise_std,
        "snr": snr,
    }


## Part 5: Batch runner

Two ways to run a batch, depending on your data:

- **`process_folder`** -- simple case: a folder of `.tcf` files, at most one run per
  file. Good for straightforward batches without a real-time-series design.
- **`process_manifest`** -- general case: an explicit list of runs, each specifying
  its own file, internal frame index, condition, oocyte ID, and real elapsed time.
  **Use this for a time-series design** like yours, where the same oocyte's 30/60/120
  min timepoints might live in 3 separate files for one oocyte, but as 3 frames inside
  one timelapse file for another -- the manifest handles both uniformly, since each row
  is just "this file, this internal frame, here's what it actually represents".

Both save an overlay image per run and append results to a CSV as they go (never
holding more than one file's data in memory), **keyed by `(file, timepoint)`** so you
can process the same file at several internal frames without overwriting results, and
skip runs already in the CSV so you can stop and resume.

**Not included here (add when ready):** downloading files from Google Drive one at a
time and deleting the local copy after processing, since many large files won't all fit
in this workspace at once. That download/delete step wraps around a call to
`process_one_file` below.

In [57]:
RESULTS_FIELDNAMES = ["file", "timepoint", "condition", "oocyte_id", "time_min",
                      "threshold_used", "mean_intensity_roi", "background_mean",
                      "background_subtracted", "noise_std", "snr"]


def append_result_row(csv_path, row):
    csv_path = Path(csv_path)
    file_exists = csv_path.exists() and csv_path.stat().st_size > 0
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_FIELDNAMES, extrasaction="ignore")
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def already_done(csv_path):
    """Set of (file, timepoint) pairs already present in the results CSV."""
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return set()
    with open(csv_path, newline="") as f:
        return {(r["file"], str(r.get("timepoint", 0))) for r in csv.DictReader(f)}


def process_one_file(file_path, results_csv, overlays_dir, timepoint=0, condition="",
                      oocyte_id="", time_min=None, outer_threshold=None, fl_channel="CH0"):
    """Process one file at one (internal) timepoint/frame end-to-end: segment, compute
    measures, save overlay, append to CSV."""
    file_path = Path(file_path)
    overlays_dir = Path(overlays_dir)
    overlays_dir.mkdir(parents=True, exist_ok=True)
    save_path = overlays_dir / f"{file_path.stem}_t{timepoint}_overlay.png"

    result = compute_snr_one_file(
        file_path, timepoint=timepoint, condition=condition, oocyte_id=oocyte_id,
        time_min=time_min, outer_threshold=outer_threshold, fl_channel=fl_channel,
        show_overlay=False, save_path=save_path, verbose=True,
    )
    append_result_row(results_csv, result)
    print(f"  -> SNR={result['snr']:.2f}, bg-sub={result['background_subtracted']:.1f}, "
          f"overlay saved to {save_path}")
    return result

def process_manifest(manifest, results_csv, overlays_dir, data_dir=".",
                      outer_threshold=None, fl_channel="CH0", skip_existing=True,
                      drive_files=None, delete_after_processing=False):
    """
    Process an explicit list of runs -- the general-purpose batch method, and the one
    to use for a real time-series design (condition x oocyte x real elapsed time),
    since it doesn't assume one run per file.

    manifest : list of dicts, one per run. Each dict must have "file" and may include:
        "timepoint"  -- internal frame index within that file (default 0)
        "condition"  -- e.g. "MTG_100" (default "")
        "oocyte_id"  -- e.g. "MTG100_oocyte1", to track the same oocyte across runs
                        (default "")
        "time_min"   -- real elapsed imaging time in minutes, e.g. 30/60/120
                        (default None)
        "outer_threshold" -- per-run threshold override (default: the outer_threshold
                        argument below, or auto via Otsu if that's also None)

    data_dir : folder the "file" entries are resolved relative to (also where files
        get downloaded to, if drive_files is given).
    drive_files : optional dict {filename: google_drive_file_id}, e.g. from
        list_drive_folder_files() (see Part 9). If a manifest entry's file isn't
        already present in data_dir, and its name is in drive_files, it's downloaded
        from Google Drive automatically before processing.
    delete_after_processing : if True, deletes the local .tcf file after it's
        successfully processed (keeps the overlay PNG and CSV row) -- use this so a
        large batch downloaded from Drive doesn't fill up local disk. Only deletes
        files that were present in drive_files (i.e. re-downloadable), never a file
        that was already sitting in data_dir before this run.
    """
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    drive_files = drive_files or {}
    existing = already_done(results_csv) if skip_existing else set()
    if existing:
        print(f"{len(existing)} (file, timepoint) pair(s) already in {results_csv} -- will skip those.")

    processed = []
    for i, entry in enumerate(manifest):
        fname = entry["file"]
        file_path = data_dir / fname
        tp = entry.get("timepoint", 0)
        cond = entry.get("condition", "")
        oid = entry.get("oocyte_id", "")
        tmin = entry.get("time_min", None)
        thresh = entry.get("outer_threshold", outer_threshold)
        key = (file_path.name, str(tp))
        if key in existing:
            print(f"[{i+1}/{len(manifest)}] Skipping {file_path.name} (timepoint {tp}, already processed)")
            continue

        downloaded_this_run = False
        if not file_path.exists():
            if fname in drive_files:
                print(f"\n[{i+1}/{len(manifest)}] {fname} not found locally -- downloading from Drive...")
                download_drive_file(drive_files[fname], file_path)
                downloaded_this_run = True
            else:
                print(f"[{i+1}/{len(manifest)}] SKIPPING {fname}: not found locally and not in drive_files")
                continue

        print(f"\n[{i+1}/{len(manifest)}] {file_path.name} (timepoint {tp}, "
              f"condition '{cond}', oocyte '{oid}', time_min={tmin})")
        try:
            process_one_file(file_path, results_csv, overlays_dir, timepoint=tp,
                              condition=cond, oocyte_id=oid, time_min=tmin,
                              outer_threshold=thresh, fl_channel=fl_channel)
            processed.append((file_path.name, tp))
        except Exception as e:
            print(f"  FAILED: {e}")
        finally:
            plt.close("all")
            if delete_after_processing and downloaded_this_run and file_path.exists():
                file_path.unlink()
                print(f"  Deleted local copy of {file_path.name} to free disk space.")

    print(f"\nDone. Processed {len(processed)} new run(s), "
          f"skipped {len(manifest) - len(processed)}.")
    return processed

   


In [58]:
%pip install filelock gdown --upgrade

Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [59]:
# Run once per environment:
# !pip install --upgrade gdown

import gdown


def list_drive_folder_files(folder_url_or_id):
    """List every file in a shared Google Drive folder ('Anyone with the link') without
    downloading anything. Returns {filename: drive_file_id}, and prints the listing.
    """
    kwargs = {"id": folder_url_or_id} if "/" not in str(folder_url_or_id) else {"url": folder_url_or_id}
    entries = gdown.download_folder(quiet=True, skip_download=True, **kwargs)
    files = {Path(e.path).name: e.id for e in entries}
    print(f"Found {len(files)} file(s):")
    for name, fid in files.items():
        print(f"  {name}  (id={fid})")
    return files


def download_drive_file(file_id, dest_path):
    """Download a single file from Google Drive by its file ID to dest_path."""
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=file_id, output=str(dest_path), quiet=False)
    return dest_path

## Part 6: Graphs

Reads the results CSV back in and plots it a few standard ways. Metric names are any of
the columns in `RESULTS_FIELDNAMES`: `threshold_used`, `mean_intensity_roi`,
`background_mean`, `background_subtracted`, `noise_std`, `snr`.

- **`plot_metric_bar`** -- one bar per (file, timepoint), sorted descending. Good
  overview figure; also works on `metric="threshold_used"` as a QC check that
  auto-thresholding behaved consistently across the batch.
- **`plot_metric_histogram`** -- distribution of a metric across the whole batch.
- **`plot_metric_vs_metric`** -- scatter of one metric against another, e.g. to sanity
  check that `snr` and `background_subtracted` broadly agree.
- **`plot_metric_over_timepoints`** -- for any file processed at more than one
  timepoint, plots that metric across timepoints as a line -- use this to show a trend
  within a timelapse.

In [60]:
def load_results_rows(csv_path):
    """Read the results CSV back in, converting numeric fields to float/int."""
    numeric_fields = ["threshold_used", "mean_intensity_roi", "background_mean",
                       "background_subtracted", "noise_std", "snr"]
    rows = []
    with open(csv_path, newline="") as f:
        for r in csv.DictReader(f):
            row = dict(r)
            for k in numeric_fields:
                if row.get(k) not in (None, ""):
                    try:
                        row[k] = float(row[k])
                    except ValueError:
                        pass
            if row.get("timepoint") not in (None, ""):
                try:
                    row["timepoint"] = int(row["timepoint"])
                except ValueError:
                    pass
            if row.get("time_min") not in (None, ""):
                try:
                    row["time_min"] = float(row["time_min"])
                except ValueError:
                    pass
            else:
                row["time_min"] = None
            rows.append(row)
    return rows


def _row_label(r):
    return f"{Path(r['file']).stem}_t{r.get('timepoint', 0)}"


def plot_metric_bar(csv_path, metric="snr", sort=True, save_path=None):
    rows = load_results_rows(csv_path)
    labels = [_row_label(r) for r in rows]
    values = [r[metric] for r in rows]
    if sort:
        order = np.argsort(values)[::-1]
        labels = [labels[i] for i in order]
        values = [values[i] for i in order]
    fig, ax = plt.subplots(figsize=(max(6, 0.4 * len(labels)), 5))
    ax.bar(range(len(values)), values, color="teal")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} per file/timepoint")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


def plot_metric_histogram(csv_path, metric="snr", bins=15, save_path=None):
    rows = load_results_rows(csv_path)
    values = [r[metric] for r in rows]
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.hist(values, bins=bins, color="slateblue", edgecolor="black")
    ax.set_xlabel(metric)
    ax.set_ylabel("count")
    ax.set_title(f"Distribution of {metric} across batch (n={len(values)})")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


def plot_metric_vs_metric(csv_path, metric_x="background_subtracted", metric_y="snr",
                           save_path=None):
    rows = load_results_rows(csv_path)
    x = [r[metric_x] for r in rows]
    y = [r[metric_y] for r in rows]
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(x, y, color="darkorange", edgecolor="black")
    ax.set_xlabel(metric_x)
    ax.set_ylabel(metric_y)
    ax.set_title(f"{metric_y} vs {metric_x}")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


def plot_metric_over_timepoints(csv_path, metric="snr", save_path=None):
    """Groups rows by base filename and plots metric vs timepoint as a line -- only
    meaningful for files that were processed at more than one timepoint."""
    rows = load_results_rows(csv_path)
    by_file = {}
    for r in rows:
        by_file.setdefault(r["file"], []).append(r)

    fig, ax = plt.subplots(figsize=(7, 5))
    any_plotted = False
    for fname, group in by_file.items():
        if len(group) < 2:
            continue
        group = sorted(group, key=lambda r: r.get("timepoint", 0))
        tps = [r.get("timepoint", 0) for r in group]
        vals = [r[metric] for r in group]
        ax.plot(tps, vals, marker="o", label=fname)
        any_plotted = True

    if not any_plotted:
        print("No file has more than one timepoint in this CSV -- nothing to plot. "
              "Process the same file at multiple timepoints first (see Part 5).")
        plt.close(fig)
        return None

    ax.set_xlabel("timepoint")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} over timepoints")
    ax.legend(fontsize=7)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


## Part 7: Comparing conditions (e.g. different probes/concentrations)

For comparing a metric **across different conditions in different oocytes** (e.g.
MitoTracker Green 100/50/25nM vs TMRM 125/60nM) -- give each file a `condition` label
via `process_one_file(..., condition="MTG_100")` or the `conditions` dict in
`process_folder`, then use the functions below.

**Note on 0-1 normalization:** these functions plot/summarize the raw metric values
rather than rescaling to 0-1. SNR is already a background/noise-normalized ratio, which
is what makes it comparable across different probes in the first place -- rescaling it
again to a fixed 0-1 range per batch would make the comparison depend on whichever
sample happens to be highest/lowest in your specific batch, rather than reflecting real
differences between conditions. If your probes were imaged with different
laser power/gain/exposure settings, keep that in mind as a caveat when interpreting
differences between conditions -- it's a limitation of the comparison, not something
normalization would fix.

In [61]:
def summarize_by_group(csv_path, metric="snr", group_field="condition"):
    """Print and return mean, std, and n of `metric` for each value of `group_field`."""
    rows = load_results_rows(csv_path)
    groups = {}
    for r in rows:
        groups.setdefault(r.get(group_field, ""), []).append(r[metric])

    summary = {}
    print(f"{metric} by {group_field}:")
    for g in sorted(groups):
        vals = np.array(groups[g])
        mean, std, n = vals.mean(), vals.std(ddof=1) if len(vals) > 1 else 0.0, len(vals)
        summary[g] = {"mean": float(mean), "std": float(std), "n": n}
        label = g if g else "(no condition set)"
        print(f"  {label:20s}  mean={mean:.3f}  std={std:.3f}  n={n}")
    return summary


def plot_metric_by_group(csv_path, metric="snr", group_field="condition", save_path=None):
    """Box plot of `metric` split by `group_field`, with individual points overlaid so
    small-n groups (a handful of oocytes per condition) are still visible."""
    rows = load_results_rows(csv_path)
    groups = {}
    for r in rows:
        groups.setdefault(r.get(group_field, ""), []).append(r[metric])

    labels = sorted(groups)
    data = [groups[g] for g in labels]

    fig, ax = plt.subplots(figsize=(max(6, 1.2 * len(labels)), 5))
    ax.boxplot(data, tick_labels=labels, showmeans=True)
    rng = np.random.default_rng(0)
    for i, vals in enumerate(data, start=1):
        jitter = rng.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, color="darkorange",
                   edgecolor="black", zorder=3, alpha=0.8)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} by {group_field}")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


## Part 8: How a measure changes over real time

For your design -- e.g. MTG 100nM imaged at 30/60/120 min in 3 different oocytes --
these plots use `time_min` (the real elapsed time you supplied) on the x-axis, and use
`oocyte_id` to keep each oocyte's own trajectory separate, regardless of whether that
oocyte's timepoints came from 3 separate files or 3 frames in one timelapse file.

- **`plot_time_series_by_oocyte`** -- one line per oocyte, colored by condition. Shows
  individual variability between oocytes -- useful to eyeball before deciding on stats
  (e.g. whether a mixed-effects/repeated-measures model is warranted given how much
  oocytes vary from each other).
- **`plot_time_series_by_condition`** -- one line per condition, showing the mean +/-
  SEM across oocytes at each timepoint. This is usually the headline figure -- "does
  MTG_100 behave differently from TMRM_60 over time".

Both require `time_min` and `oocyte_id` to have been set when you processed the files
(Part 5).

In [62]:
def plot_time_series_by_oocyte(csv_path, metric="snr", save_path=None):
    """One line per oocyte (grouped by oocyte_id), metric vs time_min, colored by
    condition. Shows individual oocyte trajectories."""
    rows = [r for r in load_results_rows(csv_path) if r.get("time_min") is not None]
    if not rows:
        print("No rows have time_min set -- nothing to plot. Pass time_min when "
              "processing files (Part 5).")
        return None

    by_oocyte = {}
    for r in rows:
        by_oocyte.setdefault(r.get("oocyte_id", ""), []).append(r)

    conditions = sorted({r.get("condition", "") for r in rows})
    cmap = plt.get_cmap("tab10")
    color_by_condition = {c: cmap(i % 10) for i, c in enumerate(conditions)}

    fig, ax = plt.subplots(figsize=(7, 5))
    seen_conditions = set()
    for oid, group in by_oocyte.items():
        group = sorted(group, key=lambda r: r["time_min"])
        tmins = [r["time_min"] for r in group]
        vals = [r[metric] for r in group]
        cond = group[0].get("condition", "")
        color = color_by_condition.get(cond, "gray")
        label = cond if cond not in seen_conditions else None
        seen_conditions.add(cond)
        ax.plot(tmins, vals, marker="o", color=color, alpha=0.8, label=label)

    ax.set_xlabel("time (min)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} over time, per oocyte")
    ax.legend(title="condition", fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


def plot_time_series_by_condition(csv_path, metric="snr", save_path=None):
    """One line per condition: mean +/- SEM across oocytes, metric vs time_min."""
    rows = [r for r in load_results_rows(csv_path) if r.get("time_min") is not None]
    if not rows:
        print("No rows have time_min set -- nothing to plot. Pass time_min when "
              "processing files (Part 5).")
        return None

    by_cond_time = {}
    for r in rows:
        key = (r.get("condition", ""), r["time_min"])
        by_cond_time.setdefault(key, []).append(r[metric])

    conditions = sorted({r.get("condition", "") for r in rows})
    cmap = plt.get_cmap("tab10")

    fig, ax = plt.subplots(figsize=(7, 5))
    for i, cond in enumerate(conditions):
        tmins = sorted({t for (c, t) in by_cond_time if c == cond})
        means, sems = [], []
        for t in tmins:
            vals = np.array(by_cond_time[(cond, t)])
            means.append(vals.mean())
            sems.append(vals.std(ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0)
        ax.errorbar(tmins, means, yerr=sems, marker="o", capsize=4,
                    color=cmap(i % 10), label=cond)

    ax.set_xlabel("time (min)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} over time, by condition (mean \u00b1 SEM)")
    ax.legend(title="condition", fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return fig


## Example usage

**Step 1 -- check what timepoints a file has** (especially for a new/unfamiliar
timelapse):

In [63]:
# list_available_timepoints("myfile.tcf")


**Step 2 -- single-file test.** Timepoints are 0-indexed, so "image 4" of a timelapse
is `timepoint=3`. Always check the overlay before trusting a batch:

In [64]:
# result = compute_snr_one_file("myfile.tcf", timepoint=3)
# print(result)


**Step 3 -- batch.** All `.tcf` files directly inside `data_dir`; give a `timepoints`
dict for any file that shouldn't use the default (`timepoint=0`):

In [65]:
# processed = process_folder(
#     data_dir="batch_files",
#     results_csv="snr_results.csv",
#     overlays_dir="overlays",
#     timepoints={"sample1.tcf": 3, "sample2.tcf": 1},   # files not listed default to timepoint 0
#     conditions={"sample1.tcf": "MTG_100", "sample2.tcf": "TMRM_60"},  # for Part 7 group comparison
# )


**Step 4 -- graphs**, once you have a results CSV:

In [66]:
# plot_metric_bar("snr_results.csv", metric="snr")
# plot_metric_histogram("snr_results.csv", metric="snr")
# plot_metric_vs_metric("snr_results.csv", metric_x="background_subtracted", metric_y="snr")
# plot_metric_over_timepoints("snr_results.csv", metric="snr")


**Step 5 -- compare conditions** (e.g. MTG 100/50/25nM vs TMRM 125/60nM), once files have `condition` labels:

In [67]:
# summarize_by_group("snr_results.csv", metric="snr")
# plot_metric_by_group("snr_results.csv", metric="snr")


**Step 6 -- time-series design** (condition x oocyte x real elapsed time), using
`process_manifest` instead of `process_folder`:

In [79]:
manifest = [
    {"file": "260810.103439.OPT007-009_25nM_MitG.007.Group1.A1.S004.TCF", "timepoint": 0,
     "condition": "MTG_25", "oocyte_id": "MTG25_S004"},
    {"file": "260810.104028.OPT007-009_25nM_MitG.008.Group1.A1.S005.TCF", "timepoint": 0,
     "condition": "MTG_25", "oocyte_id": "MTG25_S005"},
    {"file": "260810.104425.OPT007-009_25nM_MitG.009.Group1.A1.S006.TCF", "timepoint": 0,
     "condition": "MTG_25", "oocyte_id": "MTG25_S006"},
]

processed = process_manifest(
    manifest, results_csv="snr_results.csv", overlays_dir="overlays",
    data_dir="batch_files", drive_files=drive_files, delete_after_processing=True,
)


[1/3] 260810.103439.OPT007-009_25nM_MitG.007.Group1.A1.S004.TCF (timepoint 0, condition 'MTG_25', oocyte 'MTG25_S004', time_min=None)
  auto threshold: 1.3376
  24 candidates, chosen size=359521
  -> SNR=4.01, bg-sub=35.6, overlay saved to overlays/260810.103439.OPT007-009_25nM_MitG.007.Group1.A1.S004_t0_overlay.png

[2/3] 260810.104028.OPT007-009_25nM_MitG.008.Group1.A1.S005.TCF (timepoint 0, condition 'MTG_25', oocyte 'MTG25_S005', time_min=None)
  auto threshold: 1.3376
  34 candidates, chosen size=328990
  -> SNR=2.04, bg-sub=24.2, overlay saved to overlays/260810.104028.OPT007-009_25nM_MitG.008.Group1.A1.S005_t0_overlay.png

[3/3] 260810.104425.OPT007-009_25nM_MitG.009.Group1.A1.S006.TCF (timepoint 0, condition 'MTG_25', oocyte 'MTG25_S006', time_min=None)
  auto threshold: 1.3376
  22 candidates, chosen size=309607
  -> SNR=0.14, bg-sub=23.1, overlay saved to overlays/260810.104425.OPT007-009_25nM_MitG.009.Group1.A1.S006_t0_overlay.png

Done. Processed 3 new run(s), skipped 0.


In [69]:
# plot_time_series_by_oocyte("snr_results.csv", metric="snr")
# plot_time_series_by_condition("snr_results.csv", metric="snr")


In [70]:
   drive_files = list_drive_folder_files("https://drive.google.com/drive/folders/17gFkfOopA5UGYndClzDAZZLmsPFRfV7q?usp=drive_link")

Found 3 file(s):
  260810.103439.OPT007-009_25nM_MitG.007.Group1.A1.S004.TCF  (id=1wPIdtfSTC526qijBETJA6w3HkAguReuY)
  260810.104028.OPT007-009_25nM_MitG.008.Group1.A1.S005.TCF  (id=1Piiw-lc1cKF5rOaKU4mEsOQWZfommOtO)
  260810.104425.OPT007-009_25nM_MitG.009.Group1.A1.S006.TCF  (id=1IqcmUf0kSPhZ6Z-ox5i0AeV7Uh9loUwi)


In [71]:
processed = process_manifest(
    manifest, results_csv="snr_results.csv", overlays_dir="overlays",
    data_dir="batch_files", drive_files=drive_files, delete_after_processing=True,
)

3 (file, timepoint) pair(s) already in snr_results.csv -- will skip those.
[1/3] Skipping 260810.103439.OPT007-009_25nM_MitG.007.Group1.A1.S004.TCF (timepoint 0, already processed)
[2/3] Skipping 260810.104028.OPT007-009_25nM_MitG.008.Group1.A1.S005.TCF (timepoint 0, already processed)
[3/3] Skipping 260810.104425.OPT007-009_25nM_MitG.009.Group1.A1.S006.TCF (timepoint 0, already processed)

Done. Processed 0 new run(s), skipped 3.


In [72]:
print(load_results_rows("snr_results.csv"))

[{'file': '260810.103439.OPT007-009_25nM_MitG.007.Group1.A1.S004.TCF', 'timepoint': 0, 'condition': 'MTG_25', 'oocyte_id': 'MTG25_S004', 'time_min': None, 'threshold_used': 1.3376, 'mean_intensity_roi': 66.30807495117188, 'background_mean': 30.708354949951172, 'background_subtracted': 35.5997200012207, 'noise_std': 8.88774585723877, 'snr': 4.005483569517903}, {'file': '260810.104028.OPT007-009_25nM_MitG.008.Group1.A1.S005.TCF', 'timepoint': 0, 'condition': 'MTG_25', 'oocyte_id': 'MTG25_S005', 'time_min': None, 'threshold_used': 1.3376, 'mean_intensity_roi': 59.45785140991211, 'background_mean': 35.259281158447266, 'background_subtracted': 24.198570251464844, 'noise_std': 11.845863342285156, 'snr': 2.042786545163432}, {'file': '260810.104425.OPT007-009_25nM_MitG.009.Group1.A1.S006.TCF', 'timepoint': 0, 'condition': 'MTG_25', 'oocyte_id': 'MTG25_S006', 'time_min': None, 'threshold_used': 1.3376, 'mean_intensity_roi': 56.39531707763672, 'background_mean': 33.24831008911133, 'background_su

In [77]:
import pandas as pd
pd.read_csv("snr_results.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'snr_results.csv'

In [ ]:
%pip install pandas

In [78]:
import os
print(os.getcwd())
print([f for f in os.listdir(".") if f.endswith(".csv")])

/var/www/filebrowser/.projects/e2f8f835-6519-4927-ae29-9bc85b97d663
[]
